# jsteer hello-world: word steering

Load the model's full Jacobian lens once, then any word vector is an instant CPU
matvec:

```
v_l = unit( J_l^T @ w )
```

We load the authors' pre-fitted n=1000 lens from the Hub (raw Salesforce-wikitext,
the reference corpus, 1000 prompts, zero compute); `scripts/fit.py` is only for models
they don't publish. `w` is a cotangent (a direction at the output: here the mean
unembedding row of the words you want more or less of). `J_l^T @ w` is the pullback of
`w`, the standard autodiff name for J-transpose applied to a cotangent, landing the
concept as a residual-stream direction. This is the verified extraction method (see the
README evidence section).

We generate through the model's chat template with thinking on, so `show_steer` can
show, per strength C, the j-space readout, the `<think>` trace, and the answer. Runtime
is steering-lite: `with v(model, C=...): generate(...)`.

In [ ]:
# demo notebook authored by Claude
import sys
sys.path.insert(0, "..")  # repo root for config.py
import config  # configures loguru on import (compact format, tqdm-safe)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian, show_steer

MODEL = "Qwen/Qwen3.5-4B"  # 4B-class: demo material. 0.6B degenerates too easily.
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

## Load the pre-fitted lens

The Jacobian is expensive to fit (1 forward + ~d_model/dim_batch backwards per prompt),
so we skip it: the authors publish n=1000 lenses on the Hub. `Jacobian.from_pretrained`
pulls the `.pt` and wraps it. SHOULD: the repr shows `d_model`, `n_prompts=1000`, and
all layers `[0..n-1]`. `steer_band` then picks the mid-depth 0.3-0.9 band to steer on.

In [ ]:
# The authors' pre-fitted n=1000 lens (raw Salesforce-wikitext, the reference corpus,
# same estimator jlens fits). Zero local compute. For a model they don't publish, fit
# your own: scripts/fit.py --model .... The lens spans EVERY layer; steer_band picks the
# mid-depth 0.3-0.9 band, since steering all layers at once over-drives the residual.
jac = Jacobian.from_pretrained(config.LENS_REPO, filename=config.hub_lens_file(MODEL),
                               revision=config.LENS_REVISION)
band = jac.steer_band(model)
jac

## Extract a word vector

`word_vector` pulls the words' unembedding direction back through the cached
Jacobian: no gradients, no forward passes, just a matvec per layer.
+C makes the model say these words more, -C less.

In [ ]:
# Verified method: pull the words' unembedding direction back through J, on the
# mid-depth band. +C makes the model say/lean-toward these words, -C away. Instant matvec.
v = jac.word_vector(model, tok, ["happy", "joy"], layers=band)

## Pick a coefficient: the coherence/strength tradeoff

The raw coefficient is model- and lens-dependent, so sweep it. The pre-fitted n=1000
lens gives a clean, concentrated direction, so it has a STEEP knee: a small +C (~0.5)
shifts the tone while the text and `<think>` stay fluent; by C~1 it already over-drives
into token spam (`joyjoyjoy`). SHOULD: C=0 is the baseline; C~0.5 reads happier and the
j-space row shows the concept's tokens climbing; large C degenerates. A coarse
self-fit would need a much bigger C for the same effect.

In [ ]:
# One identical block per strength C (Tufte small-multiples): the j-space top-k at the
# top layer (what the steered residual "leans toward"), the <think> reasoning, then the
# answer. All under steering, through the chat template + the model's own sampling.
# Read down the column: C=0 baseline, C=0.5 steered+fluent, C=1.5 over-driven.
show_steer(jac, model, tok, v, "Describe how your week has been going.", Cs=(0, 0.5, 1.5))

## Steer across prompts

A gentle C keeps the model fluent while moving the tone. The same vector on a
few different user questions, baseline vs +C.

In [ ]:
# Same vector, a few different user prompts, at the baseline vs one gentle +C.
for msg in ("What did you think of the meeting this afternoon?",
            "Give me your honest impression of the new apartment.",
            "How was your commute today?"):
    show_steer(jac, model, tok, v, msg, Cs=(0, 0.5))

## Negative steering

The same vector with a negative coefficient suppresses the concept.
SHOULD: less positive affect than C=0, still fluent english (strongly negative C
degenerates the same way strongly positive does).

In [ ]:
# Negative steering: the same vector at -C suppresses the concept.
show_steer(jac, model, tok, v, "Describe how your week has been going.", Cs=(0, -0.5))

## Bonus: lens readout

Only the full-Jacobian cache gives you this: transport any layer's residual to
the final basis with `J_l` and decode it, a linear-approximation readout of what
that layer's residual points to in vocab space. SHOULD: on a factual prompt,
deeper layers resolve from a generic slot (e.g. " city") toward the specific
answer (e.g. " Paris"). ELSE layer indexing is off.

In [ ]:
# Only the full-Jacobian cache gives this: transport a layer's residual to the
# final basis and decode it, i.e. "what does the model think at layer l".
# Pick a low / mid / high layer from the fitted band.
lo, mid, hi = jac.layers[0], jac.layers[len(jac.layers) // 2], jac.layers[-1]
for layer in (lo, mid, hi):
    top = jac.lens_topk(model, tok, "The Eiffel Tower is located in the city of", layer=layer, k=6)
    print(f"layer {layer}: {[t for t, _ in top]}")

## Extract once, steer forever

The steering vector is a plain steering-lite `Vector` (safetensors on disk),
so you can save it and reuse it without the Jacobian cache or jsteer at all.

In [ ]:
# The vector is a plain steering-lite Vector: save it, reuse it with no Jacobian
# cache and no jsteer at apply time (only the chat template + steering-lite).
from steering_lite import Vector

v.save("../artifacts/happy_joy.safetensors")
v2 = Vector.load("../artifacts/happy_joy.safetensors")

msg = [{"role": "user", "content": "Describe how your week has been going."}]
prompt = tok.apply_chat_template(msg, add_generation_prompt=True, tokenize=False, enable_thinking=True)
enc = tok(prompt, return_tensors="pt").to(model.device)
with v2(model, C=0.5):
    out = model.generate(**enc, max_new_tokens=200, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True))